In [2]:
from urllib.request import urlopen
from bs4 import BeautifulSoup
import re
import pymysql
from random import shuffle

In [ ]:
conn = pymysql.Connection(host='127.0.0.1',
                         user='root',
                         password='MySQL%958#treebeard',
                         database='wikipedia')

""" Connect to database """
try:
    cur = conn.cursor()
    print("Connection succeeded")
except connector.Error as err:
    print(f"Error connecting to MySQL: {err}")

cur.execute('USE wikipedia')

def insertPageIfNotExists(url):
    """ 
    Search pages table for url. 
    If url does not exist in the table, add the url to the table.
    """
    cur.execute("SELECT id FROM pages WHERE url = %s LIMIT 1", (url))
    
    # Retrieve the first matching row
    page = cur.fetchone()
    if not page:
        cur.execute("INSERT INTO pages (url) VALUES (%s)", (url))
        conn.commit()
        return cur.lastrowid
    else:
        return page[0]

def loadPages():
    """
    Select all urls from the pages table
    """
    cur.execute("SELECT url FROM pages")
    return [row[0] for row in cur.fetchall()]

def insertLink(fromPageId, toPageId):
    """
    Search links table for link
    If link does not exist, add a new link to the table
    """
    cur.execute(
        "SELECT EXISTS(SELECT 1 FROM links WHERE fromPageId = %s\
         AND toPageId = %s)"
        , (int(fromPageId), int(toPageId)))
    if not cur.fetchone()[0]:
        cur.execute(
            "INSERT INTO links (fromPageId, toPageId) VALUES (%s, %s)"
            ,(int(fromPageId), int(toPageId)))
        conn.commit()

def pageHasLinks(pageId):
    """
    Checks whether a fromPageId has any links in the links table 
    """
    
    cur.execute(
        "SELECT EXISTS(SELECT 1 FROM links WHERE fromPageId = %s)"
        , int(pageId))
    return cur.fetchone()[0]

def getLinks(pageUrl, recursionLevel, pages):
    """
    Starting at a wikipedia page, pull all wikipedia href links.
    For the webpage searching for list, check to see if the page is in the pabes table.
    Add the page if it did not exist. 
    Search for all links in the webpage using BeautifulSoup
    For each links, check in the links table to see if it exists.
    Add those links that did not exist.
    For each link on the page repeat the process of getting links
    """

    if recursionLevel > 1:
        return
    pageId = insertPageIfNotExists(pageUrl)
    html = urlopen(f'http://en.wikipedia.org{pageUrl}') 
    bs = BeautifulSoup(html, 'html.parser')
    links = bs.findAll('a', href=re.compile('^(/wiki/)((?!:).)*$'))
    links = [link.attrs['href'] for link in links]

    for link in links:
        linkId = insertPageIfNotExists(link)
        insertLink(pageId, linkId)
        if not pageHasLinks(linkId):
            print(f'Getting {link}')
            pages.append(link)
            getLinks(link, recursionLevel+1, pages)
        else:
            print(f'Already fetched {link}')

getLinks('/wiki/Kevin_Bacon', 0, loadPages())

if cur:
    cur.close()
    print("Cursor closed")
if conn:
    conn.close()
    print("Database connection closed")
    

In [17]:
conn = pymysql.Connection(host='127.0.0.1',
                         user='root',
                         password='MySQL%958#treebeard',
                         database='wikipedia')

""" Connect to database """
try:
    cur = conn.cursor()
    print("Connection succeeded")
except connector.Error as err:
    print(f"Error connecting to MySQL: {err}")

cur.execute('USE wikipedia')

max_pages_to_add = 100
pages_added_counter = 0

def insertPageIfNotExists(url):
    """ 
    Search pages table for url. 
    If url does not exist in the table, add the url to the table.
    """
    global pages_added_counter # create counter of number of pages added

    # Retrieve the first matching row
    cur.execute("SELECT id FROM pages WHERE url = %s LIMIT 1", (url))
    page = cur.fetchone()
    
    if pages_added_counter >= max_pages_to_add:
        print(f"{max_pages_to_add} pages have been added, which is the limit.")
        return pages_added_counter
    elif not page:
            cur.execute("INSERT INTO pages (url) VALUES (%s)", (url))
            conn.commit()
            pages_added_counter += 1
            return cur.lastrowid
    else:
        return page[0]

def loadPages():
    """
    Select all urls from the pages table
    """
    cur.execute("SELECT url FROM pages")
    return [row[0] for row in cur.fetchall()]

def insertLink(fromPageId, toPageId):
    """
    Search links table for link
    If link does not exist, add a new link to the table
    """
    cur.execute(
        "SELECT EXISTS(SELECT 1 FROM links WHERE fromPageId = %s\
         AND toPageId = %s)"
        , (int(fromPageId), int(toPageId)))
    if not cur.fetchone()[0]:
        cur.execute(
            "INSERT INTO links (fromPageId, toPageId) VALUES (%s, %s)"
            ,(int(fromPageId), int(toPageId)))
        conn.commit()

def pageHasLinks(pageId):
    """
    Checks whether a fromPageId has any links in the links table 
    """
    
    cur.execute(
        "SELECT EXISTS(SELECT 1 FROM links WHERE fromPageId = %s)"
        , int(pageId))
    return cur.fetchone()[0]

def getLinks(pageUrl, recursionLevel, pages):
    """
    Starting at a wikipedia page, pull all wikipedia href links.
    For the webpage searching for list, check to see if the page is in the pabes table.
    Add the page if it did not exist. 
    Search for all links in the webpage using BeautifulSoup
    For each links, check in the links table to see if it exists.
    Add those links that did not exist.
    For each link on the page repeat the process of getting links
    """
    global pages_added_counter
    
    if recursionLevel > 1:
        return
    pageId = insertPageIfNotExists(pageUrl)
    html = urlopen(f'http://en.wikipedia.org{pageUrl}') 
    bs = BeautifulSoup(html, 'html.parser')
    links = bs.findAll('a', href=re.compile('^(/wiki/)((?!:).)*$'))
    links = [link.attrs['href'] for link in links]

    for link in links:
        linkId = insertPageIfNotExists(link)
        if linkId >= max_pages_to_add:
            return
        else:
            insertLink(pageId, linkId)
            if not pageHasLinks(linkId):
                print(f'Getting {link}')
                pages.append(link)
                getLinks(link, recursionLevel+1, pages)
            else:
                print(f'Already fetched {link}')

getLinks('/wiki/Kevin_Bacon', 0, loadPages())

if cur:
    cur.close()
    print("Cursor closed")
if conn:
    conn.close()
    print("Database connection closed")
    

Connection succeeded
Already fetched /wiki/Main_Page
Already fetched /wiki/Main_Page
Already fetched /wiki/Kevin_Bacon
Already fetched /wiki/Kevin_Bacon
Already fetched /wiki/Kevin_Bacon
Getting /wiki/Kevin_Bacon_(disambiguation)
Already fetched /wiki/Main_Page
Already fetched /wiki/Main_Page
Already fetched /wiki/Kevin_Bacon_(disambiguation)
Already fetched /wiki/Kevin_Bacon_(disambiguation)
Already fetched /wiki/Kevin_Bacon_(disambiguation)
Already fetched /wiki/Kevin_Bacon
Getting /wiki/Kevin_Bacon_(producer)
Getting /wiki/Kevin_Bacon_(politician)
Getting /wiki/Kevin_Bacon_(equestrian)
Getting /wiki/Tribeca_Festival
Already fetched /wiki/Main_Page
Already fetched /wiki/Main_Page
Already fetched /wiki/Tribeca_Festival
Already fetched /wiki/Tribeca_Festival
Already fetched /wiki/Tribeca_Festival
Getting /wiki/New_York_City
Getting /wiki/Film_festival
Getting /wiki/Tribeca_Enterprises
Getting /wiki/New_York_City
Getting /wiki/Robert_De_Niro
Getting /wiki/Jane_Rosenthal
Getting /wiki/Cr